# BraTS2020 Brain Tumor Segmentation - Comprehensive EDA

This notebook provides a complete exploratory data analysis of the BraTS2020 brain tumor segmentation dataset.

## Contents
1. Dataset Overview & Configuration
2. Data Loading & Structure
3. Statistical Analysis
4. Class Distribution & Balance
5. Image Quality & Intensity Analysis
6. Segmentation Mask Analysis
7. Spatial Distribution Analysis
8. Data Quality Checks
9. Sample Visualization
10. Summary & Recommendations

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import lmdb
import pickle
import torch
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 11

# Add project to path
project_root = Path('../braintumnet/src').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Dataset Overview & Configuration

In [ ]:
# Dataset paths
DATA_ROOT = Path('../braintumnet/data')
LMDB_ROOT = DATA_ROOT / 'lmdb_processed_multiclass_full'
PROCESSED_ROOT = DATA_ROOT / 'processed_multiclass_full'

# Verify paths
print("Dataset Paths:")
print(f"  LMDB Root: {LMDB_ROOT} [{'✓' if LMDB_ROOT.exists() else '✗'}]")
print(f"  Processed Root: {PROCESSED_ROOT} [{'✓' if PROCESSED_ROOT.exists() else '✗'}]")
print()

if not LMDB_ROOT.exists():
    raise FileNotFoundError(f"LMDB dataset not found at {LMDB_ROOT}")

# Dataset configuration
config = {
    'modalities': ['flair', 't1', 't1ce', 't2'],
    'num_classes': 3,  # Background, TC (Tumor Core), ED (Edema)
    'class_names': ['Background', 'Tumor Core (TC)', 'Edema (ED)'],
    'img_size': 256,
    'slices_per_case': 155,
    'num_folds': 5
}

print("Dataset Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## 2. Data Loading & Structure

In [ ]:
# Load metadata from LMDB
print("Loading LMDB metadata...")
env = lmdb.open(str(LMDB_ROOT), readonly=True, lock=False, readahead=False, meminit=False)

with env.begin() as txn:
    metadata = pickle.loads(txn.get(b'__metadata__'))
    
env.close()

print(f"\nLMDB Metadata:")
print(f"  Total samples: {metadata['num_samples']:,}")
print(f"  Slice IDs count: {len(metadata['slice_ids']):,}")
print(f"  Sample slice IDs: {metadata['slice_ids'][:5]}")

In [ ]:
# Load CSV files
all_slices_df = pd.read_csv(LMDB_ROOT / 'all_slices.csv')
labels_df = pd.read_csv(LMDB_ROOT / 'labels.csv')
mapping_df = pd.read_csv(LMDB_ROOT / 'mapping.csv')

# Rename conflicting columns to avoid merge issues
# all_slices_df['label'] = slice-level labels (Normal, Tumor, etc.)
# labels_df['label'] = case-level grade labels (0=LGG, 1=HGG)
labels_df = labels_df.rename(columns={'label': 'grade'})

print("All Slices DataFrame:")
print(f"  Shape: {all_slices_df.shape}")
print(f"  Columns: {list(all_slices_df.columns)}")
print()
all_slices_df.head(10)

In [ ]:
print("Labels DataFrame:")
print(f"  Shape: {labels_df.shape}")
print(f"  Columns: {list(labels_df.columns)}")
labels_df.head(10)

In [ ]:
print("Mapping DataFrame:")
print(f"  Shape: {mapping_df.shape}")
print(f"  Columns: {list(mapping_df.columns)}")
mapping_df.head(10)

In [ ]:
# Load fold splits
fold_data = {}
for fold in range(5):
    train_df = pd.read_csv(LMDB_ROOT / f'train_fold{fold}.csv')
    val_df = pd.read_csv(LMDB_ROOT / f'val_fold{fold}.csv')
    fold_data[fold] = {
        'train': train_df,
        'val': val_df,
        'train_size': len(train_df),
        'val_size': len(val_df)
    }

print("Fold Splits:")
for fold, data in fold_data.items():
    total = data['train_size'] + data['val_size']
    train_pct = 100 * data['train_size'] / total
    val_pct = 100 * data['val_size'] / total
    print(f"  Fold {fold}: Train={data['train_size']:,} ({train_pct:.1f}%), Val={data['val_size']:,} ({val_pct:.1f}%)")

## 3. Statistical Analysis

In [ ]:
# Merge dataframes for comprehensive analysis
# all_slices_df already has case_id, just merge with grade labels
df = all_slices_df.copy()
df = df.merge(labels_df, on='case_id', how='left')

print("Combined DataFrame:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")
print()
df.head()

In [ ]:
print("Basic Statistics:")
print(f"  Total slices: {len(df):,}")
print(f"  Unique cases: {df['case_id'].nunique():,}")
print(f"  Avg slices per case: {len(df) / df['case_id'].nunique():.1f}")

In [ ]:
# Case-level statistics
case_stats = df.groupby('case_id').agg({
    'slice_id': 'count',
    'grade': 'first'
}).rename(columns={'slice_id': 'num_slices'})

print("Case-level Statistics:")
print(case_stats['num_slices'].describe())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribution of slices per case
axes[0].hist(case_stats['num_slices'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Slices per Case')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Slices per Case')
axes[0].axvline(case_stats['num_slices'].mean(), color='red', linestyle='--', label=f"Mean: {case_stats['num_slices'].mean():.1f}")
axes[0].legend()

# Box plot
axes[1].boxplot(case_stats['num_slices'], vert=True)
axes[1].set_ylabel('Number of Slices')
axes[1].set_title('Box Plot: Slices per Case')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Class Distribution & Balance

In [ ]:
# Label distribution (grade: LGG vs HGG)
label_counts = labels_df['grade'].value_counts().sort_index()
label_names = {0: 'LGG (Low Grade)', 1: 'HGG (High Grade)'}

print("Grade Distribution (Case-level):")
for grade, count in label_counts.items():
    pct = 100 * count / len(labels_df)
    print(f"  {label_names[grade]}: {count:,} ({pct:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
bars = axes[0].bar([label_names[i] for i in label_counts.index], label_counts.values, 
                   edgecolor='black', alpha=0.7, color=['#3498db', '#e74c3c'])
axes[0].set_ylabel('Number of Cases')
axes[0].set_title('Case Grade Distribution')
axes[0].grid(True, alpha=0.3, axis='y')
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}\n({100*height/len(labels_df):.1f}%)',
                ha='center', va='bottom')

# Pie chart
colors = ['#3498db', '#e74c3c']
axes[1].pie(label_counts.values, labels=[label_names[i] for i in label_counts.index], 
           autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Case Grade Distribution (Proportion)')

plt.tight_layout()
plt.show()

In [ ]:
# Label distribution across folds
fold_label_dist = []

for fold in range(5):
    train_df = fold_data[fold]['train']
    val_df = fold_data[fold]['val']
    
    # Extract case IDs (fold CSVs already have case_id column)
    train_cases = train_df['case_id'].unique()
    val_cases = val_df['case_id'].unique()
    
    # Get labels (using grade column)
    train_labels = labels_df[labels_df['case_id'].isin(train_cases)]['grade']
    val_labels = labels_df[labels_df['case_id'].isin(val_cases)]['grade']
    
    fold_label_dist.append({
        'fold': fold,
        'train_lgg': (train_labels == 0).sum(),
        'train_hgg': (train_labels == 1).sum(),
        'val_lgg': (val_labels == 0).sum(),
        'val_hgg': (val_labels == 1).sum()
    })

fold_dist_df = pd.DataFrame(fold_label_dist)
print("Label Distribution Across Folds:")
fold_dist_df

In [ ]:
# Visualize fold distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

x = np.arange(5)
width = 0.35

# Train distribution
axes[0].bar(x - width/2, fold_dist_df['train_lgg'], width, label='LGG', alpha=0.7, color='#3498db')
axes[0].bar(x + width/2, fold_dist_df['train_hgg'], width, label='HGG', alpha=0.7, color='#e74c3c')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Number of Cases')
axes[0].set_title('Train Set Label Distribution Across Folds')
axes[0].set_xticks(x)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Val distribution
axes[1].bar(x - width/2, fold_dist_df['val_lgg'], width, label='LGG', alpha=0.7, color='#3498db')
axes[1].bar(x + width/2, fold_dist_df['val_hgg'], width, label='HGG', alpha=0.7, color='#e74c3c')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Number of Cases')
axes[1].set_title('Validation Set Label Distribution Across Folds')
axes[1].set_xticks(x)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Image Quality & Intensity Analysis

In [ ]:
# Sample random images for intensity analysis
print("Sampling images for intensity analysis...")
np.random.seed(42)
sample_size = min(1000, metadata['num_samples'])
sample_indices = np.random.choice(metadata['num_samples'], sample_size, replace=False)

# Initialize storage
intensity_stats = defaultdict(list)
modality_names = ['FLAIR', 'T1', 'T1CE', 'T2']

# Open LMDB
env = lmdb.open(str(LMDB_ROOT), readonly=True, lock=False, readahead=True, meminit=False)

# Sample images
with env.begin() as txn:
    for idx in tqdm(sample_indices, desc="Analyzing intensities"):
        key = f"{idx:08d}".encode('ascii')
        sample_bytes = txn.get(key)
        
        if sample_bytes is not None:
            sample = pickle.loads(sample_bytes)
            image = sample['image']  # (4, H, W) uint8
            
            # Collect statistics for each modality
            for mod_idx in range(4):
                mod_data = image[mod_idx]
                intensity_stats[f'modality_{mod_idx}'].append({
                    'mean': float(mod_data.mean()),
                    'std': float(mod_data.std()),
                    'min': float(mod_data.min()),
                    'max': float(mod_data.max()),
                    'median': float(np.median(mod_data)),
                    'q25': float(np.percentile(mod_data, 25)),
                    'q75': float(np.percentile(mod_data, 75))
                })

env.close()

# Convert to DataFrame
intensity_dfs = {}
for mod_idx in range(4):
    intensity_dfs[mod_idx] = pd.DataFrame(intensity_stats[f'modality_{mod_idx}'])

print(f"\nAnalyzed {sample_size} images")
print("\nIntensity Statistics (across sampled images):")
for mod_idx, mod_name in enumerate(modality_names):
    print(f"\n{mod_name}:")
    print(intensity_dfs[mod_idx].describe())

In [ ]:
# Visualize intensity distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for mod_idx, mod_name in enumerate(modality_names):
    df_mod = intensity_dfs[mod_idx]
    
    # Plot distribution of means
    axes[mod_idx].hist(df_mod['mean'], bins=50, alpha=0.6, label='Mean', color='blue', edgecolor='black')
    axes[mod_idx].hist(df_mod['std'], bins=50, alpha=0.6, label='Std Dev', color='red', edgecolor='black')
    axes[mod_idx].set_xlabel('Intensity Value')
    axes[mod_idx].set_ylabel('Frequency')
    axes[mod_idx].set_title(f'{mod_name} - Intensity Distribution')
    axes[mod_idx].legend()
    axes[mod_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for intensity comparison across modalities
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Prepare data
mean_data = [intensity_dfs[i]['mean'].values for i in range(4)]
std_data = [intensity_dfs[i]['std'].values for i in range(4)]

# Box plot for means
bp1 = axes[0].boxplot(mean_data, labels=modality_names, patch_artist=True)
for patch, color in zip(bp1['boxes'], ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel('Mean Intensity')
axes[0].set_title('Mean Intensity Comparison Across Modalities')
axes[0].grid(True, alpha=0.3, axis='y')

# Box plot for std
bp2 = axes[1].boxplot(std_data, labels=modality_names, patch_artist=True)
for patch, color in zip(bp2['boxes'], ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Standard Deviation')
axes[1].set_title('Intensity Std Dev Comparison Across Modalities')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Segmentation Mask Analysis

In [ ]:
# Sample masks for analysis
print("Analyzing segmentation masks...")
np.random.seed(42)
mask_sample_size = min(2000, metadata['num_samples'])
mask_sample_indices = np.random.choice(metadata['num_samples'], mask_sample_size, replace=False)

# Initialize storage
mask_stats = []
class_pixel_counts = {0: [], 1: [], 2: []}  # BG, TC, ED

# Open LMDB
env = lmdb.open(str(LMDB_ROOT), readonly=True, lock=False, readahead=True, meminit=False)

# Analyze masks
with env.begin() as txn:
    for idx in tqdm(mask_sample_indices, desc="Analyzing masks"):
        key = f"{idx:08d}".encode('ascii')
        sample_bytes = txn.get(key)
        
        if sample_bytes is not None:
            sample = pickle.loads(sample_bytes)
            mask = sample['mask']  # (H, W) uint8
            
            # Count pixels per class
            unique, counts = np.unique(mask, return_counts=True)
            class_counts = dict(zip(unique, counts))
            
            total_pixels = mask.size
            
            # Store statistics
            for cls in range(3):
                count = class_counts.get(cls, 0)
                class_pixel_counts[cls].append(count)
            
            # Calculate tumor presence
            has_tc = 1 in class_counts
            has_ed = 2 in class_counts
            has_tumor = has_tc or has_ed
            
            mask_stats.append({
                'bg_pixels': class_counts.get(0, 0),
                'tc_pixels': class_counts.get(1, 0),
                'ed_pixels': class_counts.get(2, 0),
                'bg_ratio': class_counts.get(0, 0) / total_pixels,
                'tc_ratio': class_counts.get(1, 0) / total_pixels,
                'ed_ratio': class_counts.get(2, 0) / total_pixels,
                'has_tumor': has_tumor,
                'has_tc': has_tc,
                'has_ed': has_ed
            })

env.close()

# Convert to DataFrame
mask_stats_df = pd.DataFrame(mask_stats)

print(f"\nAnalyzed {len(mask_stats_df)} masks")
print("\nMask Statistics:")
print(mask_stats_df.describe())

In [ ]:
# Class balance in masks
print("\nClass Balance (pixel-level):")
print(f"  Background ratio: {mask_stats_df['bg_ratio'].mean():.4f} ± {mask_stats_df['bg_ratio'].std():.4f}")
print(f"  Tumor Core ratio: {mask_stats_df['tc_ratio'].mean():.4f} ± {mask_stats_df['tc_ratio'].std():.4f}")
print(f"  Edema ratio: {mask_stats_df['ed_ratio'].mean():.4f} ± {mask_stats_df['ed_ratio'].std():.4f}")

print("\nTumor Presence:")
print(f"  Slices with tumor: {mask_stats_df['has_tumor'].sum():,} ({100*mask_stats_df['has_tumor'].mean():.1f}%)")
print(f"  Slices with TC: {mask_stats_df['has_tc'].sum():,} ({100*mask_stats_df['has_tc'].mean():.1f}%)")
print(f"  Slices with ED: {mask_stats_df['has_ed'].sum():,} ({100*mask_stats_df['has_ed'].mean():.1f}%)")
print(f"  Slices without tumor: {(~mask_stats_df['has_tumor']).sum():,} ({100*(~mask_stats_df['has_tumor']).mean():.1f}%)")

In [ ]:
# Visualize class ratios
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Class ratio distributions
axes[0, 0].hist(mask_stats_df['bg_ratio'], bins=50, alpha=0.7, color='gray', edgecolor='black')
axes[0, 0].set_xlabel('Background Ratio')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Background Pixel Ratio Distribution')
axes[0, 0].axvline(mask_stats_df['bg_ratio'].mean(), color='red', linestyle='--', label=f"Mean: {mask_stats_df['bg_ratio'].mean():.3f}")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(mask_stats_df['tc_ratio'], bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0, 1].set_xlabel('Tumor Core Ratio')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Tumor Core Pixel Ratio Distribution')
axes[0, 1].axvline(mask_stats_df['tc_ratio'].mean(), color='red', linestyle='--', label=f"Mean: {mask_stats_df['tc_ratio'].mean():.3f}")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(mask_stats_df['ed_ratio'], bins=50, alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].set_xlabel('Edema Ratio')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Edema Pixel Ratio Distribution')
axes[1, 0].axvline(mask_stats_df['ed_ratio'].mean(), color='red', linestyle='--', label=f"Mean: {mask_stats_df['ed_ratio'].mean():.3f}")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Tumor presence pie chart
tumor_counts = [mask_stats_df['has_tumor'].sum(), (~mask_stats_df['has_tumor']).sum()]
axes[1, 1].pie(tumor_counts, labels=['With Tumor', 'No Tumor'], autopct='%1.1f%%', 
              colors=['#e74c3c', '#95a5a6'], startangle=90)
axes[1, 1].set_title('Tumor Presence in Sampled Slices')

plt.tight_layout()
plt.show()

In [ ]:
# Class imbalance analysis
mean_bg_ratio = mask_stats_df['bg_ratio'].mean()
mean_tc_ratio = mask_stats_df['tc_ratio'].mean()
mean_ed_ratio = mask_stats_df['ed_ratio'].mean()

# Calculate imbalance ratios
imbalance_ratios = [
    mean_bg_ratio,
    mean_tc_ratio,
    mean_ed_ratio
]

# Suggested class weights (inverse frequency)
total = sum(imbalance_ratios)
suggested_weights = [total / (3 * ratio) if ratio > 0 else 0 for ratio in imbalance_ratios]

print("\nClass Imbalance Analysis:")
print(f"  Background: {mean_bg_ratio:.4f} (weight: {suggested_weights[0]:.2f})")
print(f"  Tumor Core: {mean_tc_ratio:.4f} (weight: {suggested_weights[1]:.2f})")
print(f"  Edema: {mean_ed_ratio:.4f} (weight: {suggested_weights[2]:.2f})")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

class_names_short = ['BG', 'TC', 'ED']

# Class distribution
bars = axes[0].bar(class_names_short, imbalance_ratios, color=['gray', 'blue', 'orange'], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Mean Pixel Ratio')
axes[0].set_title('Average Class Distribution (Pixel-level)')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, imbalance_ratios):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}',
                ha='center', va='bottom')

# Suggested weights
bars = axes[1].bar(class_names_short, suggested_weights, color=['gray', 'blue', 'orange'], alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Suggested Weight')
axes[1].set_title('Suggested Class Weights (Inverse Frequency)')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, suggested_weights):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}',
                ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 7. Spatial Distribution Analysis

In [ ]:
# Analyze tumor location (center of mass)
print("Analyzing spatial distribution of tumors...")
tumor_locations = []

# Sample subset for spatial analysis
np.random.seed(42)
spatial_sample_size = min(500, metadata['num_samples'])
spatial_sample_indices = np.random.choice(metadata['num_samples'], spatial_sample_size, replace=False)

env = lmdb.open(str(LMDB_ROOT), readonly=True, lock=False, readahead=True, meminit=False)

with env.begin() as txn:
    for idx in tqdm(spatial_sample_indices, desc="Computing tumor centers"):
        key = f"{idx:08d}".encode('ascii')
        sample_bytes = txn.get(key)
        
        if sample_bytes is not None:
            sample = pickle.loads(sample_bytes)
            mask = sample['mask']
            
            # Get tumor mask (TC + ED)
            tumor_mask = (mask == 1) | (mask == 2)
            
            if tumor_mask.any():
                # Calculate center of mass
                y_coords, x_coords = np.where(tumor_mask)
                center_y = y_coords.mean()
                center_x = x_coords.mean()
                
                # Normalize to [0, 1]
                h, w = mask.shape
                norm_y = center_y / h
                norm_x = center_x / w
                
                tumor_locations.append({
                    'center_x': center_x,
                    'center_y': center_y,
                    'norm_x': norm_x,
                    'norm_y': norm_y,
                    'tumor_area': int(tumor_mask.sum())
                })

env.close()

tumor_loc_df = pd.DataFrame(tumor_locations)
print(f"\nAnalyzed {len(tumor_loc_df)} slices with tumors")

if len(tumor_loc_df) > 0:
    print("\nTumor Center Statistics (normalized):")
    print(tumor_loc_df[['norm_x', 'norm_y']].describe())
else:
    print("\nNo tumors found in sampled slices")

In [ ]:
# Visualize tumor spatial distribution
if len(tumor_loc_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Scatter plot of tumor centers
    axes[0].scatter(tumor_loc_df['norm_x'], tumor_loc_df['norm_y'], alpha=0.3, s=10)
    axes[0].set_xlabel('Normalized X')
    axes[0].set_ylabel('Normalized Y')
    axes[0].set_title('Tumor Center Locations (Normalized)')
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(0, 1)
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5)
    axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.5)
    
    # 2D histogram (heatmap)
    h, xedges, yedges = np.histogram2d(tumor_loc_df['norm_x'], tumor_loc_df['norm_y'], bins=20)
    im = axes[1].imshow(h.T, origin='lower', extent=[0, 1, 0, 1], cmap='hot', aspect='auto')
    axes[1].set_xlabel('Normalized X')
    axes[1].set_ylabel('Normalized Y')
    axes[1].set_title('Tumor Center Density Heatmap')
    plt.colorbar(im, ax=axes[1], label='Frequency')
    axes[1].axhline(0.5, color='cyan', linestyle='--', alpha=0.5)
    axes[1].axvline(0.5, color='cyan', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
else:
    print("No tumor location data to visualize")

In [ ]:
# Tumor size distribution
if len(tumor_loc_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(tumor_loc_df['tumor_area'], bins=50, alpha=0.7, color='purple', edgecolor='black')
    axes[0].set_xlabel('Tumor Area (pixels)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Tumor Size Distribution')
    axes[0].axvline(tumor_loc_df['tumor_area'].mean(), color='red', linestyle='--', 
                   label=f"Mean: {tumor_loc_df['tumor_area'].mean():.0f}")
    axes[0].axvline(tumor_loc_df['tumor_area'].median(), color='blue', linestyle='--', 
                   label=f"Median: {tumor_loc_df['tumor_area'].median():.0f}")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    axes[1].boxplot(tumor_loc_df['tumor_area'], vert=True)
    axes[1].set_ylabel('Tumor Area (pixels)')
    axes[1].set_title('Tumor Size Distribution (Box Plot)')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\nTumor Size Statistics:")
    print(tumor_loc_df['tumor_area'].describe())
else:
    print("No tumor size data to visualize")

## 8. Data Quality Checks

In [ ]:
# Check for issues in the data
print("Performing data quality checks...\n")

quality_issues = []

# Check 1: Missing labels
missing_labels = mapping_df[~mapping_df['case_id'].isin(labels_df['case_id'])]
if len(missing_labels) > 0:
    quality_issues.append(f"Missing labels for {len(missing_labels)} slices")
    print(f"⚠ Missing labels for {len(missing_labels)} slices")
else:
    print("✓ All slices have corresponding labels")

# Check 2: Duplicate slice IDs
duplicate_slices = all_slices_df['slice_id'].duplicated().sum()
if duplicate_slices > 0:
    quality_issues.append(f"Found {duplicate_slices} duplicate slice IDs")
    print(f"⚠ Found {duplicate_slices} duplicate slice IDs")
else:
    print("✓ No duplicate slice IDs")

# Check 3: Case ID consistency
unique_cases_slices = mapping_df['case_id'].nunique()
unique_cases_labels = labels_df['case_id'].nunique()
if unique_cases_slices != unique_cases_labels:
    quality_issues.append(f"Case count mismatch: {unique_cases_slices} in mapping vs {unique_cases_labels} in labels")
    print(f"⚠ Case count mismatch: {unique_cases_slices} in mapping vs {unique_cases_labels} in labels")
else:
    print(f"✓ Consistent case count: {unique_cases_labels}")

# Check 4: Fold coverage
all_slices_in_folds = set()
for fold in range(5):
    train_slices = set(fold_data[fold]['train']['slice_id'])
    val_slices = set(fold_data[fold]['val']['slice_id'])
    all_slices_in_folds.update(train_slices)
    all_slices_in_folds.update(val_slices)

total_slices = set(all_slices_df['slice_id'])
missing_from_folds = total_slices - all_slices_in_folds
if len(missing_from_folds) > 0:
    quality_issues.append(f"{len(missing_from_folds)} slices not in any fold")
    print(f"⚠ {len(missing_from_folds)} slices not in any fold")
else:
    print("✓ All slices covered by folds")

# Check 5: Fold overlap
fold_overlap = False
for fold in range(5):
    train_slices = set(fold_data[fold]['train']['slice_id'])
    val_slices = set(fold_data[fold]['val']['slice_id'])
    overlap = train_slices & val_slices
    if len(overlap) > 0:
        fold_overlap = True
        quality_issues.append(f"Fold {fold}: {len(overlap)} slices in both train and val")
        print(f"⚠ Fold {fold}: {len(overlap)} slices in both train and val")

if not fold_overlap:
    print("✓ No overlap between train and val in any fold")

# Summary
print(f"\n{'='*50}")
if len(quality_issues) == 0:
    print("✓ All quality checks passed!")
else:
    print(f"⚠ Found {len(quality_issues)} quality issues:")
    for issue in quality_issues:
        print(f"  - {issue}")

## 9. Sample Visualization

In [ ]:
# Visualize random samples
print("Loading sample images for visualization...")
np.random.seed(42)

# Select samples with tumors if possible
tumor_indices = [i for i in range(len(mask_stats)) if mask_stats[i]['has_tumor']]
if len(tumor_indices) >= 6:
    vis_sample_indices = np.random.choice(tumor_indices, 6, replace=False)
    # Map back to actual LMDB indices
    vis_lmdb_indices = [mask_sample_indices[i] for i in vis_sample_indices]
else:
    vis_lmdb_indices = np.random.choice(metadata['num_samples'], min(6, metadata['num_samples']), replace=False)

# Load samples
samples = []
env = lmdb.open(str(LMDB_ROOT), readonly=True, lock=False, readahead=True, meminit=False)

with env.begin() as txn:
    for idx in vis_lmdb_indices:
        key = f"{idx:08d}".encode('ascii')
        sample_bytes = txn.get(key)
        
        if sample_bytes is not None:
            sample = pickle.loads(sample_bytes)
            samples.append(sample)

env.close()

print(f"Loaded {len(samples)} samples for visualization")

In [ ]:
# Visualize samples
if len(samples) > 0:
    fig, axes = plt.subplots(len(samples), 5, figsize=(20, 4*len(samples)))
    if len(samples) == 1:
        axes = axes.reshape(1, -1)
    
    for i, sample in enumerate(samples):
        image = sample['image']  # (4, H, W)
        mask = sample['mask']    # (H, W)
        slice_id = sample['slice_id']
        
        # Display each modality
        for j, mod_name in enumerate(['FLAIR', 'T1', 'T1CE', 'T2']):
            axes[i, j].imshow(image[j], cmap='gray')
            axes[i, j].set_title(f'{mod_name}')
            axes[i, j].axis('off')
        
        # Display mask with overlay
        axes[i, 4].imshow(image[0], cmap='gray', alpha=0.7)  # FLAIR as background
        
        # Overlay mask
        mask_colored = np.zeros((*mask.shape, 4))
        mask_colored[mask == 1] = [0, 0, 1, 0.5]  # TC: Blue
        mask_colored[mask == 2] = [1, 0.5, 0, 0.5]  # ED: Orange
        
        axes[i, 4].imshow(mask_colored)
        axes[i, 4].set_title(f'Mask\n{slice_id}')
        axes[i, 4].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No samples to visualize")

## 10. Summary & Recommendations

In [ ]:
# Generate summary report
print("="*70)
print("DATASET SUMMARY REPORT")
print("="*70)

print("\n1. DATASET OVERVIEW")
print(f"   Total cases: {labels_df['case_id'].nunique():,}")
print(f"   Total slices: {len(all_slices_df):,}")
print(f"   Avg slices/case: {len(all_slices_df) / labels_df['case_id'].nunique():.1f}")
print(f"   Image size: {config['img_size']}x{config['img_size']}")
print(f"   Modalities: {', '.join(config['modalities'])}")

print("\n2. CLASS DISTRIBUTION (Case-level)")
for grade, count in label_counts.items():
    pct = 100 * count / len(labels_df)
    print(f"   {label_names[grade]}: {count:,} ({pct:.1f}%)")

print("\n3. CLASS BALANCE (Pixel-level)")
print(f"   Background: {mask_stats_df['bg_ratio'].mean():.4f}")
print(f"   Tumor Core: {mask_stats_df['tc_ratio'].mean():.4f}")
print(f"   Edema: {mask_stats_df['ed_ratio'].mean():.4f}")

print("\n4. TUMOR STATISTICS")
print(f"   Slices with tumor: {mask_stats_df['has_tumor'].sum():,} ({100*mask_stats_df['has_tumor'].mean():.1f}%)")
if len(tumor_loc_df) > 0:
    print(f"   Avg tumor area: {tumor_loc_df['tumor_area'].mean():.0f} pixels")
    print(f"   Median tumor area: {tumor_loc_df['tumor_area'].median():.0f} pixels")

print("\n5. FOLD SPLITS")
for fold, data in fold_data.items():
    total = data['train_size'] + data['val_size']
    train_pct = 100 * data['train_size'] / total
    val_pct = 100 * data['val_size'] / total
    print(f"   Fold {fold}: Train={data['train_size']:,} ({train_pct:.1f}%), Val={data['val_size']:,} ({val_pct:.1f}%)")

print("\n6. RECOMMENDATIONS")
print("   ✓ Use class weights for loss function:")
print(f"     - Background: {suggested_weights[0]:.2f}")
print(f"     - Tumor Core: {suggested_weights[1]:.2f}")
print(f"     - Edema: {suggested_weights[2]:.2f}")
print("   ✓ Consider focal loss due to severe class imbalance")
print("   ✓ Use data augmentation (rotation, flipping, intensity)")
print("   ✓ Consider online hard example mining (OHEM)")
print(f"   ✓ Current config uses: {config['num_folds']}-fold cross-validation")
print("   ✓ LMDB backend provides 10-15x faster loading than PNG")

print("\n" + "="*70)
print("END OF REPORT")
print("="*70)

## 11. Export Summary Statistics

In [ ]:
# Export summary to JSON
summary = {
    'dataset_overview': {
        'total_cases': int(labels_df['case_id'].nunique()),
        'total_slices': int(len(all_slices_df)),
        'avg_slices_per_case': float(len(all_slices_df) / labels_df['case_id'].nunique()),
        'image_size': config['img_size'],
        'modalities': config['modalities'],
        'num_classes': config['num_classes']
    },
    'class_distribution_case': {
        label_names[grade]: {
            'count': int(count),
            'percentage': float(100 * count / len(labels_df))
        } for grade, count in label_counts.items()
    },
    'class_balance_pixel': {
        'background': float(mask_stats_df['bg_ratio'].mean()),
        'tumor_core': float(mask_stats_df['tc_ratio'].mean()),
        'edema': float(mask_stats_df['ed_ratio'].mean())
    },
    'suggested_class_weights': {
        'background': float(suggested_weights[0]),
        'tumor_core': float(suggested_weights[1]),
        'edema': float(suggested_weights[2])
    },
    'tumor_statistics': {
        'slices_with_tumor': int(mask_stats_df['has_tumor'].sum()),
        'tumor_percentage': float(100 * mask_stats_df['has_tumor'].mean()),
        'avg_tumor_area': float(tumor_loc_df['tumor_area'].mean()) if len(tumor_loc_df) > 0 else 0.0,
        'median_tumor_area': float(tumor_loc_df['tumor_area'].median()) if len(tumor_loc_df) > 0 else 0.0
    },
    'fold_splits': {
        f'fold_{fold}': {
            'train_size': int(data['train_size']),
            'val_size': int(data['val_size']),
            'train_percentage': float(100 * data['train_size'] / (data['train_size'] + data['val_size'])),
            'val_percentage': float(100 * data['val_size'] / (data['train_size'] + data['val_size']))
        } for fold, data in fold_data.items()
    }
}

# Save to file
output_path = Path('../braintumnet/data/eda_summary.json')
with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary exported to: {output_path}")